In [ ]:
# import numpy as np

# # Function
# def f(x):
#     return np.exp(-x**2)

# # Trapezoid rule
# def trapezoid(f, a, b, n):
#     x = np.linspace(a, b, n+1)
#     y = f(x)
#     dx = (b - a)/n
#     return (dx/2) * np.sum(y[:-1] + y[1:])

# # Adaptive trapezoid
# def adaptive_trapezoid(f, a, b, tol=1e-10):
#     n = 2
#     prev = trapezoid(f, a, b, n)
#     steps = [(n, prev)]
#     while True:
#         n *= 2
#         curr = trapezoid(f, a, b, n)
#         steps.append((n, curr))
#         if abs(curr - prev) < tol:
#             return curr, steps
#         prev = curr

# # Parameters
# a, b = 0, 1
# I_exact = 0.746824132812  # reference value

# # Fixed trapezoid results
# n_values = [2, 4, 8, 16, 32, 64, 128, 256]
# print("=== Trapezoid Rule (Fixed n) ===")
# for n in n_values:
#     I_n = trapezoid(f, a, b, n)
#     error = abs(I_n - I_exact)
#     print(f"n={n:4d} | Result={I_n:.12f} | Error={error:.2e}")

# # Adaptive trapezoid results
# print("\n=== Adaptive Trapezoid Refinement ===")
# I_adapt, steps = adaptive_trapezoid(f, a, b, tol=1e-10)
# for n, val in steps:
#     error = abs(val - I_exact)
#     print(f"n={n:4d} | Result={val:.12f} | Error={error:.2e}")

# print(f"\nFinal Adaptive Result = {I_adapt:.12f}")
# print(f"Exact Analytical Value = {I_exact:.12f}")


In [ ]:
import ROOT
import numpy as np

def RosenBrock(vecx):
    x = vecx[0]
    y = vecx[1]
    return (1 - x)**2 + 100 * (y - x**2)**2  # Correct Rosenbrock function

# create minimizer giving a name and a name (optionally) for the specific algorithm
#  possible choices are:
#     minimizerName                  algoName
#
#     Minuit                     Migrad, Simplex,Combined,Scan  (default is Migrad)
#     Minuit2                    Migrad, BFGS, Simplex,Combined,Scan  (default is Migrad)
#     GSLMultiMin                ConjugateFR, ConjugatePR, BFGS, BFGS2, SteepestDescent
#     GSLSimAn
#     Genetic

def NumericalMinimization(minimizerName="Minuit2",
                          algoName="Migrad",
                          randomSeed=-1):
    
    minimizer = ROOT.Math.Factory.CreateMinimizer(minimizerName, algoName)
    if (not minimizer):
        raise RuntimeError(
            "Cannot create minimizer \"{}\". Maybe the required library was not built?".format(minimizerName))

    # Set tolerance and other minimizer parameters, one can also use default
    # values

    minimizer.SetMaxFunctionCalls(1000000)  # working for Minuit/Minuit2
    # for GSL minimizers - no effect in Minuit/Minuit2
    minimizer.SetMaxIterations(10000)
    minimizer.SetTolerance(0.001)
    minimizer.SetPrintLevel(1)

    # Create function wrapper for minimizer

    f = ROOT.Math.Functor(RosenBrock, 2)

    # Starting point
    variable = [-1., 1.2]
    step = [0.01, 0.01]
    if (randomSeed >= 0):
        r = ROOT.TRandom2(randomSeed)
        variable[0] = r.Uniform(-20, 20)
        variable[1] = r.Uniform(-20, 20)

    minimizer.SetFunction(f)

    # Set the free variables to be minimized !
    minimizer.SetVariable(0, "x", variable[0], step[0])
    minimizer.SetVariable(1, "y", variable[1], step[1])

    # Do the minimization
    ret = minimizer.Minimize()

    xs = minimizer.X()
    print("Minimum: f({} , {}) = {}".format(xs[0], xs[1], minimizer.MinValue()))

    # Expected minimum is f(1,1) = 0
    expected_min = 0.0
    tolerance = 1.E-4
    if (ret and abs(minimizer.MinValue() - expected_min) < tolerance):
        print("Minimizer {} - {} converged to the right minimum!".format(minimizerName, algoName))
    else:
        print("Minimizer {} - {} failed to converge! Found minimum at f({}, {}) = {}".format(
            minimizerName, algoName, xs[0], xs[1], minimizer.MinValue()))
        # Don't raise an error for demonstration, just print warning

if __name__ == "__main__":
    NumericalMinimization()

In [ ]:
# import numpy as np
# from scipy.optimize import curve_fit
# from scipy.stats import t

# # Example model
# def model(x, a, b, c, d):
#     return a * np.exp(-b * x) + c * x + d

# # Fake data
# xdata = np.linspace(0, 4, 50)
# y = model(xdata, 2.5, 1.3, 0.5, 1.0)
# rng = np.random.default_rng(0)
# y_noise = y + 0.2 * rng.normal(size=len(xdata))

# # Fit
# popt, pcov = curve_fit(model, xdata, y_noise)

# # Extract standard errors
# perr = np.sqrt(np.diag(pcov))

# # 90% confidence intervals
# alpha = 0.10
# dof = max(0, len(xdata) - len(popt))  # degrees of freedom
# tval = t.ppf(1.0 - alpha/2., dof)

# ci = [ (p - tval*e, p + tval*e) for p, e in zip(popt, perr) ]

# print("Parameters:", popt)
# print("90% CI:", ci)


In [ ]:
# import numpy as np
# from iminuit import Minuit
# from iminuit.cost import LeastSquares

# # Example model
# def model(x, a, b, c, d):
#     return a * np.exp(-b * x) + c * x + d

# # Fake data
# x = np.linspace(0, 4, 50)
# y = model(x, 2.5, 1.3, 0.5, 1.0) + 0.2 * np.random.normal(size=len(x))
# yerr = np.full_like(y, 0.2)

# # Least squares cost
# cost = LeastSquares(x, y, yerr, model)
# m = Minuit(cost, a=1, b=1, c=1, d=1)  # initial guesses
# m.migrad()  # minimize
# m.hesse()   # covariance matrix

# print(m.values)   # best-fit params
# print(m.errors)   # 1σ errors (~68%)

# # For 90% CI, use m.mnprofile or m.mncontour
# for name in m.parameters:
#     ci90 = m.draw_mnprofile(name)  # 90% ≈ 1.64σ
#     # print(name, "90% CI:", ci90)


In [ ]:
# import ROOT
# import numpy as np

# # Sample data for chi-squared calculation
# # You can replace this with your actual data
# x_data = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
# y_data = np.array([2.1, 3.9, 6.1, 8.0, 9.9])
# y_errors = np.array([0.2, 0.3, 0.2, 0.4, 0.3])

# def linear_model(x, params):
#     """Linear model: y = a*x + b"""
#     a, b = params[0], params[1]
#     return a * x + b

# def chi2_function(params):
#     """
#     Calculate chi-squared for a linear fit
#     params[0] = slope (a)
#     params[1] = intercept (b)
#     """
#     chi2 = 0.0
#     for i in range(len(x_data)):
#         predicted = linear_model(x_data[i], params)
#         residual = y_data[i] - predicted
#         chi2 += (residual / y_errors[i])**2
#     return chi2

# # Alternative: Generic chi2 function for any model
# def generic_chi2_function(params):
#     """
#     Generic chi-squared function - modify this for your specific model
#     """
#     # Example: quadratic model y = a*x^2 + b*x + c
#     # Uncomment and modify as needed:
    
#     # chi2 = 0.0
#     # for i in range(len(x_data)):
#     #     predicted = params[0]*x_data[i]**2 + params[1]*x_data[i] + params[2]
#     #     residual = y_data[i] - predicted
#     #     chi2 += (residual / y_errors[i])**2
#     # return chi2
    
#     # For now, use the linear model
#     return chi2_function(params)

# def NumericalMinimization(minimizerName="Minuit2",
#                           algoName="",
#                           randomSeed=-1):
    
#     minimizer = ROOT.Math.Factory.CreateMinimizer(minimizerName, algoName)
#     if (not minimizer):
#         raise RuntimeError(
#             "Cannot create minimizer \"{}\". Maybe the required library was not built?".format(minimizerName))
    
#     # Set tolerance and other minimizer parameters
#     minimizer.SetMaxFunctionCalls(1000000)
#     minimizer.SetMaxIterations(10000)
#     minimizer.SetTolerance(0.001)
#     minimizer.SetPrintLevel(1)
    
#     # Create function wrapper for minimizer
#     # Use 2 parameters for linear fit (slope and intercept)
#     f = ROOT.Math.Functor(chi2_function, 2)
    
#     # Starting point for parameters [slope, intercept]
#     # You can estimate these from your data or use reasonable guesses
#     variable = [1.0, 0.0]  # Initial guess: slope=1, intercept=0
#     step = [0.01, 0.01]    # Step sizes for parameters
    
#     if (randomSeed >= 0):
#         r = ROOT.TRandom2(randomSeed)
#         variable[0] = r.Uniform(-10, 10)  # Random slope
#         variable[1] = r.Uniform(-10, 10)  # Random intercept
    
#     minimizer.SetFunction(f)
    
#     # Set the free variables to be minimized
#     minimizer.SetVariable(0, "slope", variable[0], step[0])
#     minimizer.SetVariable(1, "intercept", variable[1], step[1])
    
#     # Optional: Set parameter limits if needed
#     # minimizer.SetVariableLimits(0, -100, 100)  # Limit slope
#     # minimizer.SetVariableLimits(1, -100, 100)  # Limit intercept
    
#     # Do the minimization
#     ret = minimizer.Minimize()
    
#     xs = minimizer.X()
#     chi2_min = minimizer.MinValue()
    
#     print("Minimum: chi2(slope={:.4f}, intercept={:.4f}) = {:.4f}".format(
#         xs[0], xs[1], chi2_min))
    
#     # Calculate degrees of freedom
#     ndf = len(x_data) - 2  # n_data_points - n_parameters
#     reduced_chi2 = chi2_min / ndf
    
#     print("Reduced chi2 = {:.4f}".format(reduced_chi2))
#     print("Number of degrees of freedom = {}".format(ndf))
    
#     # Check convergence
#     if ret:
#         print("Minimizer {} - {} converged successfully!".format(minimizerName, algoName))
        
#         # Print parameter errors if available
#         if minimizer.Errors():
#             errors = minimizer.Errors()
#             print("Parameter errors:")
#             print("  slope error = {:.4f}".format(errors[0]))
#             print("  intercept error = {:.4f}".format(errors[1]))
#     else:
#         print("Minimizer {} - {} failed to converge !!!".format(minimizerName, algoName))
#         raise RuntimeError("NumericalMinimization failed to converge!")
    
#     return xs, chi2_min, reduced_chi2

# def plot_results(params):
#     """Optional: Plot the data and fitted curve"""
#     try:
#         import matplotlib.pyplot as plt
        
#         # Generate points for the fitted line
#         x_fit = np.linspace(min(x_data), max(x_data), 100)
#         y_fit = linear_model(x_fit, params)
        
#         plt.figure(figsize=(8, 6))
#         plt.errorbar(x_data, y_data, yerr=y_errors, fmt='o', label='Data')
#         plt.plot(x_fit, y_fit, 'r-', label=f'Fit: y = {params[0]:.3f}x + {params[1]:.3f}')
#         plt.xlabel('x')
#         plt.ylabel('y')
#         plt.legend()
#         plt.grid(True)
#         plt.title('Chi-squared Fit Results')
#         plt.show()
        
#     except ImportError:
#         print("matplotlib not available for plotting")

# if __name__ == "__main__":
#     # Run the minimization
#     fitted_params, min_chi2, reduced_chi2 = NumericalMinimization()
    
#     # Optionally plot results
#     # plot_results(fitted_params)

In [ ]:
import ROOT
import math
import numpy as np

def generic_function(x, par):
    """
    Generic function to fit
    f(x) = par[0] * sin(par[1]*x) + par[2] * exp(par[3]*x) + par[4]
    """
    return par[0] * math.sin(par[1] * x[0]) + par[2] * math.exp(par[3] * x[0]) + par[4]

def create_sample_data():
    """Create sample data with some noise"""
    x = np.linspace(0, 10, 50)
    # True function: f(x) = 2.5 * sin(0.8*x) + 1.2 * exp(-0.3*x) + 0.5
    y_true = 2.5 * np.sin(0.8*x) + 1.2 * np.exp(-0.3*x) + 0.5
    # Add some Gaussian noise
    y_data = y_true + 0.3 * np.random.normal(size=len(x))
    return x, y_data, y_true

def main():
    # Create sample data
    x_data, y_data, y_true = create_sample_data()
    
    # Create a ROOT graph with the data
    graph = ROOT.TGraphErrors(len(x_data))
    
    for i, (x, y) in enumerate(zip(x_data, y_data)):
        graph.SetPoint(i, x, y)
        graph.SetPointError(i, 0, 0.3)  # Assuming constant error of 0.3
    
    # Create the fit function using TF1
    fit_function = ROOT.TF1("fit_function", generic_function, 0, 10, 5)
    
    # Set initial parameter values and names
    fit_function.SetParName(0, "Amplitude")
    fit_function.SetParName(1, "Frequency")
    fit_function.SetParName(2, "ExpAmplitude")
    fit_function.SetParName(3, "ExpDecay")
    fit_function.SetParName(4, "Offset")
    
    # Set initial parameter guesses
    fit_function.SetParameter(0, 2.0)   # Amplitude guess
    fit_function.SetParameter(1, 1.0)   # Frequency guess
    fit_function.SetParameter(2, 1.0)   # Exponential amplitude guess
    fit_function.SetParameter(3, -0.5)  # Exponential decay guess
    fit_function.SetParameter(4, 0.0)   # Offset guess
    
    # Set parameter limits (optional but often helpful)
    fit_function.SetParLimits(0, 0.1, 5.0)    # Amplitude between 0.1 and 5
    fit_function.SetParLimits(1, 0.1, 2.0)    # Frequency between 0.1 and 2
    fit_function.SetParLimits(3, -2.0, 0.0)   # Decay rate negative
    
    # Perform the least squares fit
    print("Performing least squares fit...")
    fit_result = graph.Fit(fit_function, "S")  # "S" saves the fit result
    
    # Print fit results
    print("\n=== Fit Results ===")
    print(f"Fit status: {fit_result.Status()}")
    print(f"Chi-squared: {fit_function.GetChisquare():.4f}")
    print(f"NDF: {fit_function.GetNDF()}")
    print(f"Chi-squared/NDF: {fit_function.GetChisquare()/fit_function.GetNDF():.4f}")
    
    print("\n=== Parameter Values ===")
    for i in range(5):
        par_name = fit_function.GetParName(i)
        par_value = fit_function.GetParameter(i)
        par_error = fit_function.GetParError(i)
        print(f"{par_name}: {par_value:.4f} ± {par_error:.4f}")
    
    # Create a canvas and draw the results
    canvas = ROOT.TCanvas("canvas", "Least Squares Fit Example", 800, 600)
    
    # Draw the graph and fit
    graph.SetTitle("Least Squares Fit with Generic Function")
    graph.GetXaxis().SetTitle("x")
    graph.GetYaxis().SetTitle("y")
    graph.SetMarkerStyle(20)
    graph.SetMarkerSize(0.8)
    graph.Draw("AP")
    
    # Draw the fit function
    fit_function.SetLineColor(ROOT.kRed)
    fit_function.SetLineWidth(2)
    fit_function.Draw("same")
    
    # Add a legend
    legend = ROOT.TLegend(0.7, 0.7, 0.9, 0.9)
    legend.AddEntry(graph, "Data", "p")
    legend.AddEntry(fit_function, "Fit", "l")
    legend.Draw()
    
    canvas.Update()
    canvas.Draw()

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Exemplo de uso do método LeastSquareFit() da classe ROOT::Fit::Fitter
usando PyROOT para fazer um ajuste de mínimos quadrados em dados binados.
"""

import ROOT
import numpy as np

def main():
    # Configuração do estilo
    ROOT.gStyle.SetOptFit(1111)
    ROOT.gStyle.SetOptStat(1111)
    
    # 1. Criar dados simulados (distribuição gaussiana + ruído)
    print("=" * 60)
    print("Exemplo de Least Square Fit com ROOT::Fit::Fitter")
    print("=" * 60)
    
    # Parâmetros da gaussiana verdadeira
    true_mean = 5.0
    true_sigma = 1.5
    true_amplitude = 1000.0
    
    # Criar histograma para armazenar os dados
    nbins = 50
    xmin, xmax = 0.0, 10.0
    h1 = ROOT.TH1D("h1", "Dados para Least Square Fit;x;Contagens", 
                   nbins, xmin, xmax)
    
    # Preencher o histograma com dados gaussianos
    np.random.seed(42)
    data = np.random.normal(true_mean, true_sigma, int(true_amplitude))
    for value in data:
        h1.Fill(value)
    
    print(f"\nDados gerados:")
    print(f"  - Média verdadeira: {true_mean}")
    print(f"  - Sigma verdadeiro: {true_sigma}")
    print(f"  - Número de eventos: {h1.GetEntries()}")
    
    # 2. Preparar o BinData para o fit
    # Converter o histograma em um objeto BinData
    data_range = ROOT.Fit.DataRange(xmin, xmax)
    data_options = ROOT.Fit.DataOptions()
    
    bin_data = ROOT.Fit.BinData(data_options, data_range)
    ROOT.Fit.FillData(bin_data, h1)
    
    print(f"\nBinData criado com {bin_data.Size()} pontos")
    
    # 3. Definir a função modelo (gaussiana)
    # f(x) = p[0] * exp(-0.5 * ((x - p[1]) / p[2])^2)
    func = ROOT.TF1("gaus", "gaus", xmin, xmax)
    
    # Valores iniciais dos parâmetros
    func.SetParameter(0, 100.0)   # Amplitude inicial
    func.SetParameter(1, 5.0)     # Média inicial
    func.SetParameter(2, 1.0)     # Sigma inicial
    
    # Criar um wrapper IParamFunction
    wrapped_func = ROOT.Math.WrappedTF1(func)
    
    print("\nParâmetros iniciais:")
    for i in range(func.GetNpar()):
        print(f"  p[{i}] = {func.GetParameter(i):.4f}")
    
    # 4. Configurar e executar o Fitter
    fitter = ROOT.Fit.Fitter()
    
    # Configurar o fit
    fitter.Config().SetMinimizer("Minuit2", "Migrad")
    fitter.Config().MinimizerOptions().SetPrintLevel(1)
    
    # Definir a função modelo
    fitter.SetFunction(wrapped_func, False)
    
    # Executar o Least Square Fit
    print("\n" + "=" * 60)
    print("Executando LeastSquareFit...")
    print("=" * 60)
    
    fit_result = fitter.LeastSquareFit(bin_data)
    
    # 5. Exibir resultados
    print("\n" + "=" * 60)
    print("RESULTADOS DO FIT")
    print("=" * 60)
    
    if fit_result:
        result = fitter.Result()
        print(f"\nStatus do fit: {'Sucesso' if result.IsValid() else 'Falhou'}")
        print(f"Chi2/NDF: {result.Chi2():.4f} / {result.Ndf()}")
        print(f"Chi2/NDF reduzido: {result.Chi2()/result.Ndf():.4f}")
        print(f"Probabilidade: {result.Prob():.6f}")
        
        print("\nParâmetros ajustados:")
        for i in range(result.NPar()):
            print(f"  p[{i}] = {result.Parameter(i):.4f} ± {result.ParError(i):.4f}")
        
        # Atualizar a função TF1 com os parâmetros ajustados
        for i in range(result.NPar()):
            func.SetParameter(i, result.Parameter(i))
            func.SetParError(i, result.ParError(i))
        
        # 6. Visualizar o resultado
        # canvas = ROOT.TCanvas("canvas", "Least Square Fit Result", 800, 600)
        # canvas.SetGrid()
        
        # h1.SetLineColor(ROOT.kBlue)
        # h1.SetLineWidth(2)
        # h1.Draw("E")
        
        # func.SetLineColor(ROOT.kRed)
        # func.SetLineWidth(2)
        # func.Draw("SAME")
        
        # # Adicionar legenda
        # legend = ROOT.TLegend(0.15, 0.65, 0.45, 0.88)
        # legend.SetBorderSize(1)
        # legend.SetFillColor(0)
        # legend.AddEntry(h1, "Dados", "lep")
        # legend.AddEntry(func, "Fit: Gaussiana", "l")
        # legend.AddEntry(ROOT.nullptr, 
        #                f"#chi^{{2}}/NDF = {result.Chi2()/result.Ndf():.3f}", "")
        # legend.AddEntry(ROOT.nullptr, 
        #                f"#mu = {result.Parameter(1):.3f} #pm {result.ParError(1):.3f}", "")
        # legend.AddEntry(ROOT.nullptr, 
        #                f"#sigma = {result.Parameter(2):.3f} #pm {result.ParError(2):.3f}", "")
        # legend.Draw()
        
        # canvas.Update()
        
        # # Salvar o resultado
        # canvas.SaveAs("leastsquare_fit_result.png")
        # print("\nGráfico salvo como: leastsquare_fit_result.png")
        
        # # Manter a janela aberta
        # print("\nPressione Enter para sair...")
        # input()
    else:
        print("\nErro: O fit falhou!")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Exemplo de uso do método LeastSquareFit() da classe ROOT::Fit::Fitter
usando PyROOT para fazer um ajuste de mínimos quadrados em dados binados.
"""

import ROOT
import numpy as np

def gaussian_model(x, par):
    """
    Função modelo: Gaussiana
    f(x) = par[0] * exp(-0.5 * ((x - par[1]) / par[2])^2)
    
    Args:
        x: array com coordenadas x
        par: array com parâmetros [amplitude, mean, sigma]
    
    Returns:
        Valor da função gaussiana
    """
    arg = 0.0
    if par[2] != 0:
        arg = (x[0] - par[1]) / par[2]
    
    return par[0] * ROOT.TMath.Exp(-0.5 * arg * arg)

def main():
    # Configuração do estilo
    ROOT.gStyle.SetOptFit(1111)
    ROOT.gStyle.SetOptStat(1111)
    
    # 1. Criar dados simulados (distribuição gaussiana + ruído)
    print("=" * 60)
    print("Exemplo de Least Square Fit com ROOT::Fit::Fitter")
    print("=" * 60)
    
    # Parâmetros da gaussiana verdadeira
    true_mean = 5.0
    true_sigma = 1.5
    true_amplitude = 1000.0
    
    # Criar histograma para armazenar os dados
    nbins = 50
    xmin, xmax = 0.0, 10.0
    h1 = ROOT.TH1D("h1", "Dados para Least Square Fit;x;Contagens", 
                   nbins, xmin, xmax)
    
    # Preencher o histograma com dados gaussianos
    np.random.seed(42)
    data = np.random.normal(true_mean, true_sigma, int(true_amplitude))
    for value in data:
        h1.Fill(value)
    
    print(f"\nDados gerados:")
    print(f"  - Média verdadeira: {true_mean}")
    print(f"  - Sigma verdadeiro: {true_sigma}")
    print(f"  - Número de eventos: {h1.GetEntries()}")
    
    # 2. Preparar o BinData para o fit
    # Converter o histograma em um objeto BinData
    data_range = ROOT.Fit.DataRange(xmin, xmax)
    data_options = ROOT.Fit.DataOptions()
    
    bin_data = ROOT.Fit.BinData(data_options, data_range)
    ROOT.Fit.FillData(bin_data, h1)
    
    print(f"\nBinData criado com {bin_data.Size()} pontos")
    
    # 3. Definir a função modelo (gaussiana)
    # f(x) = p[0] * exp(-0.5 * ((x - p[1]) / p[2])^2)
    func = ROOT.TF1("gaus", gaussian_model, xmin, xmax, 3)
    
    # Valores iniciais dos parâmetros
    func.SetParameter(0, 100.0)   # Amplitude inicial
    func.SetParameter(1, 5.0)     # Média inicial
    func.SetParameter(2, 1.0)     # Sigma inicial
    
    # Criar um wrapper IParamFunction
    wrapped_func = ROOT.Math.WrappedTF1(func)
    
    print("\nParâmetros iniciais:")
    for i in range(func.GetNpar()):
        print(f"  p[{i}] = {func.GetParameter(i):.4f}")
    
    # 4. Configurar e executar o Fitter
    fitter = ROOT.Fit.Fitter()
    
    # Configurar o fit
    fitter.Config().SetMinimizer("Minuit2", "Migrad")
    fitter.Config().MinimizerOptions().SetPrintLevel(1)
    
    # Definir a função modelo
    fitter.SetFunction(wrapped_func, False)
    
    # Executar o Least Square Fit
    print("\n" + "=" * 60)
    print("Executando LeastSquareFit...")
    print("=" * 60)
    
    fit_result = fitter.LeastSquareFit(bin_data)
    
    # 5. Exibir resultados
    print("\n" + "=" * 60)
    print("RESULTADOS DO FIT")
    print("=" * 60)
    
    if fit_result:
        result = fitter.Result()
        print(f"\nStatus do fit: {'Sucesso' if result.IsValid() else 'Falhou'}")
        print(f"Chi2/NDF: {result.Chi2():.4f} / {result.Ndf()}")
        print(f"Probabilidade: {result.Prob():.6f}")
        
        print("\nParâmetros ajustados:")
        for i in range(result.NPar()):
            print(f"  p[{i}] = {result.Parameter(i):.4f} ± {result.ParError(i):.4f}")
        
        # Atualizar a função TF1 com os parâmetros ajustados
        for i in range(result.NPar()):
            func.SetParameter(i, result.Parameter(i))
            func.SetParError(i, result.ParError(i))
        
        # 6. Visualizar o resultado
        canvas = ROOT.TCanvas("canvas", "Least Square Fit Result", 800, 600)
        canvas.SetGrid()
        
        h1.SetLineColor(ROOT.kBlue)
        h1.SetLineWidth(2)
        h1.Draw("E")
        
        func.SetLineColor(ROOT.kRed)
        func.SetLineWidth(2)
        func.Draw("SAME")
        
        # Adicionar legenda
        legend = ROOT.TLegend(0.15, 0.65, 0.45, 0.88)
        legend.SetBorderSize(1)
        legend.SetFillColor(0)
        legend.AddEntry(h1, "Dados", "lep")
        legend.AddEntry(func, "Fit: Gaussiana", "l")
        legend.AddEntry(ROOT.nullptr, 
                       f"#chi^{{2}}/NDF = {result.Chi2()/result.Ndf():.3f}", "")
        legend.AddEntry(ROOT.nullptr, 
                       f"#mu = {result.Parameter(1):.3f} #pm {result.ParError(1):.3f}", "")
        legend.AddEntry(ROOT.nullptr, 
                       f"#sigma = {result.Parameter(2):.3f} #pm {result.ParError(2):.3f}", "")
        legend.Draw()
        
        canvas.Update()
        
        # Salvar o resultado
        canvas.SaveAs("leastsquare_fit_result.png")
        print("\nGráfico salvo como: leastsquare_fit_result.png")
        
        # Manter a janela aberta
        print("\nPressione Enter para sair...")
        input()
    else:
        print("\nErro: O fit falhou!")

if __name__ == "__main__":
    main()

In [4]:
#!/usr/bin/env python3
"""
Exemplo de uso do método LeastSquareFit() da classe ROOT::Fit::Fitter
usando PyROOT para fazer um ajuste de mínimos quadrados em dados binados.
"""

import ROOT
import numpy as np

def model_function(x, par):
    """
    Função modelo com 4 parâmetros livres
    
    Args:
        x: array com coordenadas x
        par: array com parâmetros [mg, eps, a1, a2]
    
    Returns:
        Valor da função
    """
    mg = par[0]
    eps = par[1]
    a1 = par[2]
    a2 = par[3]
    
    # Exemplo de função com os 4 parâmetros
    # Você pode modificar esta expressão conforme necessário
    return a1 * ROOT.TMath.Exp(-x[0] / mg) + a2 * (x[0] ** eps)

def calculate_chi2_dof(func_model, x_data, y_data, y_errors, initial_params, xmin, xmax):
    """
    Função genérica para calcular Chi²/DOF usando LeastSquareFit
    
    Args:
        func_model: Função Python callable(x, par) a ser minimizada
        x_data: array com dados experimentais em x
        y_data: array com dados experimentais em y
        y_errors: array com erros em y
        initial_params: lista com valores iniciais dos parâmetros
        xmin: limite inferior do intervalo de ajuste
        xmax: limite superior do intervalo de ajuste
    
    Returns:
        dict: Dicionário com resultados do fit contendo:
            - 'chi2': valor do Chi²
            - 'ndf': graus de liberdade
            - 'chi2_dof': Chi²/DOF
            - 'prob': probabilidade do fit
            - 'parameters': lista com parâmetros ajustados
            - 'errors': lista com erros dos parâmetros
            - 'valid': bool indicando se o fit foi bem sucedido
    """
    # Criar TGraphErrors com os dados
    n_points = len(x_data)
    graph = ROOT.TGraphErrors(n_points)
    
    for i in range(n_points):
        graph.SetPoint(i, x_data[i], y_data[i])
        graph.SetPointError(i, 0, y_errors[i])
    
    # Preparar o BinData
    data_range = ROOT.Fit.DataRange(xmin, xmax)
    data_options = ROOT.Fit.DataOptions()
    
    bin_data = ROOT.Fit.BinData(data_options, data_range)
    ROOT.Fit.FillData(bin_data, graph)
    
    # Criar a função TF1 com o modelo fornecido
    npar = len(initial_params)
    func = ROOT.TF1("fit_func", func_model, xmin, xmax, npar)
    
    # Definir parâmetros iniciais
    for i, param in enumerate(initial_params):
        func.SetParameter(i, param)
    
    # Criar wrapper e configurar o fitter
    wrapped_func = ROOT.Math.WrappedTF1(func)
    fitter = ROOT.Fit.Fitter()
    
    fitter.Config().SetMinimizer("Minuit2", "Migrad")
    fitter.Config().MinimizerOptions().SetPrintLevel(0)
    
    fitter.SetFunction(wrapped_func, False)
    
    # Executar o fit
    fit_result = fitter.LeastSquareFit(bin_data)
    
    # Extrair resultados
    result = fitter.Result()
    
    output = {
        'chi2': result.Chi2(),
        'ndf': result.Ndf(),
        'chi2_dof': result.Chi2() / result.Ndf() if result.Ndf() > 0 else 0.0,
        'prob': result.Prob(),
        'parameters': [result.Parameter(i) for i in range(result.NPar())],
        'errors': [result.ParError(i) for i in range(result.NPar())],
        'valid': result.IsValid()
    }
    
    return output

def main():
    # Configuração do estilo
    ROOT.gStyle.SetOptFit(1111)
    ROOT.gStyle.SetOptStat(1111)
    
    # 1. Criar dados simulados (distribuição gaussiana + ruído)
    print("=" * 60)
    print("Exemplo de Least Square Fit com ROOT::Fit::Fitter")
    print("=" * 60)
    
    # Parâmetros da gaussiana verdadeira
    true_mean = 5.0
    true_sigma = 1.5
    true_amplitude = 1000.0
    
    # Criar histograma para armazenar os dados
    nbins = 50
    xmin, xmax = 0.0, 10.0
    h1 = ROOT.TH1D("h1", "Dados para Least Square Fit;x;Contagens", 
                   nbins, xmin, xmax)
    
    # Preencher o histograma com dados gaussianos
    np.random.seed(42)
    data = np.random.normal(true_mean, true_sigma, int(true_amplitude))
    for value in data:
        h1.Fill(value)
    
    print(f"\nDados gerados:")
    print(f"  - Média verdadeira: {true_mean}")
    print(f"  - Sigma verdadeiro: {true_sigma}")
    print(f"  - Número de eventos: {h1.GetEntries()}")
    
    # 2. Preparar o BinData para o fit
    # Converter o histograma em um objeto BinData
    data_range = ROOT.Fit.DataRange(xmin, xmax)
    data_options = ROOT.Fit.DataOptions()
    
    bin_data = ROOT.Fit.BinData(data_options, data_range)
    ROOT.Fit.FillData(bin_data, h1)
    
    print(f"\nBinData criado com {bin_data.Size()} pontos")
    
    # ============================================================
    # EXEMPLO DE USO DA FUNÇÃO GENÉRICA calculate_chi2_dof
    # ============================================================
    print("\n" + "=" * 60)
    print("Testando função genérica calculate_chi2_dof")
    print("=" * 60)
    
    # Extrair dados do histograma para usar na função genérica
    x_test = []
    y_test = []
    y_err_test = []
    
    for i in range(1, h1.GetNbinsX() + 1):
        x_test.append(h1.GetBinCenter(i))
        y_test.append(h1.GetBinContent(i))
        y_err_test.append(h1.GetBinError(i) if h1.GetBinContent(i) > 0 else 1.0)
    
    # Usar a função genérica
    fit_generic = calculate_chi2_dof(
        func_model=gaussian_model,
        x_data=x_test,
        y_data=y_test,
        y_errors=y_err_test,
        initial_params=[100.0, 5.0, 1.0],
        xmin=xmin,
        xmax=xmax
    )
    
    print(f"\nResultados da função genérica:")
    print(f"  Status: {'Sucesso' if fit_generic['valid'] else 'Falhou'}")
    print(f"  Chi2/NDF: {fit_generic['chi2']:.4f} / {fit_generic['ndf']}")
    print(f"  Chi2/DOF: {fit_generic['chi2_dof']:.4f}")
    print(f"  Probabilidade: {fit_generic['prob']:.6f}")
    print(f"\n  Parâmetros ajustados:")
    for i, (param, error) in enumerate(zip(fit_generic['parameters'], fit_generic['errors'])):
        print(f"    p[{i}] = {param:.4f} ± {error:.4f}")
    
    # ============================================================
    # MÉTODO ORIGINAL (para comparação)
    # ============================================================
    
    # 3. Definir a função modelo (gaussiana)
    # f(x) = p[0] * exp(-0.5 * ((x - p[1]) / p[2])^2)
    func = ROOT.TF1("gaus", gaussian_model, xmin, xmax, 3)
    
    # Valores iniciais dos parâmetros
    func.SetParameter(0, 100.0)   # Amplitude inicial
    func.SetParameter(1, 5.0)     # Média inicial
    func.SetParameter(2, 1.0)     # Sigma inicial
    
    # Criar um wrapper IParamFunction
    wrapped_func = ROOT.Math.WrappedTF1(func)
    
    print("\nParâmetros iniciais:")
    for i in range(func.GetNpar()):
        print(f"  p[{i}] = {func.GetParameter(i):.4f}")
    
    # 4. Configurar e executar o Fitter
    fitter = ROOT.Fit.Fitter()
    
    # Configurar o fit
    fitter.Config().SetMinimizer("Minuit2", "Migrad")
    fitter.Config().MinimizerOptions().SetPrintLevel(1)
    
    # Definir a função modelo
    fitter.SetFunction(wrapped_func, False)
    
    # Executar o Least Square Fit
    print("\n" + "=" * 60)
    print("Executando LeastSquareFit (método original)...")
    print("=" * 60)
    
    fit_result = fitter.LeastSquareFit(bin_data)
    
    # 5. Exibir resultados
    print("\n" + "=" * 60)
    print("RESULTADOS DO FIT (método original)")
    print("=" * 60)
    
    if fit_result:
        result = fitter.Result()
        print(f"\nStatus do fit: {'Sucesso' if result.IsValid() else 'Falhou'}")
        print(f"Chi2/NDF: {result.Chi2():.4f} / {result.Ndf()}")
        print(f"Probabilidade: {result.Prob():.6f}")
        
        print("\nParâmetros ajustados:")
        for i in range(result.NPar()):
            print(f"  p[{i}] = {result.Parameter(i):.4f} ± {result.ParError(i):.4f}")
        
        # Atualizar a função TF1 com os parâmetros ajustados
        for i in range(result.NPar()):
            func.SetParameter(i, result.Parameter(i))
            func.SetParError(i, result.ParError(i))
        
        # 6. Visualizar o resultado
        canvas = ROOT.TCanvas("canvas", "Least Square Fit Result", 800, 600)
        canvas.SetGrid()
        
        h1.SetLineColor(ROOT.kBlue)
        h1.SetLineWidth(2)
        h1.Draw("E")
        
        func.SetLineColor(ROOT.kRed)
        func.SetLineWidth(2)
        func.Draw("SAME")
        
        # Adicionar legenda
        legend = ROOT.TLegend(0.15, 0.65, 0.45, 0.88)
        legend.SetBorderSize(1)
        legend.SetFillColor(0)
        legend.AddEntry(h1, "Dados", "lep")
        legend.AddEntry(func, "Fit: Gaussiana", "l")
        legend.AddEntry(ROOT.nullptr, 
                       f"#chi^{{2}}/NDF = {result.Chi2()/result.Ndf():.3f}", "")
        legend.AddEntry(ROOT.nullptr, 
                       f"#mu = {result.Parameter(1):.3f} #pm {result.ParError(1):.3f}", "")
        legend.AddEntry(ROOT.nullptr, 
                       f"#sigma = {result.Parameter(2):.3f} #pm {result.ParError(2):.3f}", "")
        legend.Draw()
        
        canvas.Update()
        
        # Salvar o resultado
        canvas.SaveAs("leastsquare_fit_result.png")
        print("\nGráfico salvo como: leastsquare_fit_result.png")
        
        # Manter a janela aberta
        print("\nPressione Enter para sair...")
        input()
    else:
        print("\nErro: O fit falhou!")

if __name__ == "__main__":
    main()

Exemplo de Least Square Fit com ROOT::Fit::Fitter

Dados gerados:
  - Média verdadeira: 5.0
  - Sigma verdadeiro: 1.5
  - Número de eventos: 1000.0

BinData criado com 44 pontos

Testando função genérica calculate_chi2_dof

Resultados da função genérica:
  Status: Sucesso
  Chi2/NDF: 41.3367 / 47
  Chi2/DOF: 0.8795
  Probabilidade: 0.705348

  Parâmetros ajustados:
    p[0] = 54.1771 ± 2.1880
    p[1] = 5.0150 ± 0.0469
    p[2] = 1.4140 ± 0.0351

Parâmetros iniciais:
  p[0] = 100.0000
  p[1] = 5.0000
  p[2] = 1.0000

Executando LeastSquareFit (método original)...

RESULTADOS DO FIT (método original)

Status do fit: Sucesso
Chi2/NDF: 40.4338 / 41
Probabilidade: 0.495621

Parâmetros ajustados:
  p[0] = 53.9670 ± 2.2020
  p[1] = 5.0166 ± 0.0472
  p[2] = 1.4211 ± 0.0364

Gráfico salvo como: leastsquare_fit_result.png

Pressione Enter para sair...
Minuit2Minimizer: Minimize with max-calls 1345 convergence for edm < 0.01 strategy 1
Minuit2Minimizer : Valid minimum - status = 0
FVAL  = 40.433

Info in <TCanvas::Print>: png file leastsquare_fit_result.png has been created
